# Cross-Model Hidden-State Analysis — Extended Experiments

**Study 02 · Research notebook · Google Colab (GPU recommended)**

This notebook reorganizes the original exploratory notebook `Untitled7(1).ipynb` while retaining its research questions and core metrics. It explores: (1) model-to-model geometry (kNN / CKA / RSA), (2) layerwise comparisons and topic probing, (3) broader model families, (4) autoregressive versus diffusion representations, (5) neighbour consensus and lexical/topic effects, (6) resampling stability, and (7) transfer to STS-B and PAWS.

**Important:** Results are produced by running this notebook; no historical results are presented as new outputs. Some optional model downloads or external repositories can change over time. The optional nano-model section uses externally supplied model code; review that code before executing. See methodological cautions at the end.

**How to use:** Select *Runtime → Change runtime type → GPU*, then run top-to-bottom. `INCLUDE_NANO=True` enables the AR/diffusion experiment. `RUN_TINYSTORIES=True` enables the extra-dataset replication; both increase runtime. All other experiments run by default.


## 1 · Environment and reproducibility

In [ ]:
%pip -q install "transformers>=4.45,<5" "datasets>=3,<5" scipy scikit-learn pandas matplotlib sentencepiece safetensors huggingface_hub accelerate


In [ ]:
import os, gc, re, json, itertools, subprocess, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer, BertModel, BertConfig, T5EncoderModel, BartModel

SEED = 42
np.random.seed(SEED); torch.manual_seed(0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_PER_CLASS = 500          # preserve original: 2,000 AG News samples
MAX_LENGTH = 128          # same as original HF extraction
BATCH_SIZE = 32
DEPTH = 0.9
K = 10
SUBSET_N = 400           # original model-matrix / RSA / CKA comparison
INCLUDE_NANO = False     # optional external nanoDiffusionLab models
RUN_TINYSTORIES = False  # optional additional dataset replication
RESULTS_DIR = Path('study02_results'); RESULTS_DIR.mkdir(exist_ok=True)
print('Device:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'not available')


### Extraction method

Hidden states are obtained using the same Hugging Face models and **mask-weighted mean pooling** as the source notebook, with float32 storage. Batching and `torch.inference_mode()` save memory; pretrained model weights are **not** altered. The runtime executes one pretrained model at a time to limit GPU memory use. Saved representations stay in CPU RAM, while neighbourhood and matrix operations may run on the GPU. No half precision is enabled to avoid changing numerical comparisons.

In [ ]:
@torch.inference_mode()
def extract_layers(model, tokenizer, sentences, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    model = model.to(DEVICE).eval()
    pooled_layers = None
    for offset in range(0, len(sentences), batch_size):
        batch = tokenizer(sentences[offset:offset+batch_size], padding=True,
                          truncation=True, max_length=max_length, return_tensors='pt').to(DEVICE)
        # Some HF models reject token_type_ids; original loader/model pairings do not need them.
        if 'token_type_ids' in batch and getattr(model.config, 'model_type', '') in {'gpt_neox','t5','bart','distilbert'}:
            batch.pop('token_type_ids')
        result = model(**batch, output_hidden_states=True, return_dict=True)
        states = result.hidden_states
        mask = batch['attention_mask'].unsqueeze(-1).to(dtype=torch.float32)
        reps = [((h.float() * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)).cpu() for h in states]
        if pooled_layers is None: pooled_layers = [[] for _ in reps]
        for store, rep in zip(pooled_layers, reps): store.append(rep)
    model.to('cpu')
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return [torch.cat(parts, dim=0) for parts in pooled_layers]

def normalize_rep(X):
    # Deliberately preserve original standardization semantics (torch.std default).
    X = X.float()
    X = (X - X.mean(dim=0)) / (X.std(dim=0) + 1e-6)
    return F.normalize(X, dim=1)

def get_neighbors(X, k=10):
    Xg = X.to(DEVICE)
    similarity = Xg @ Xg.T
    similarity.fill_diagonal_(-float('inf'))
    out = similarity.topk(k, dim=1).indices.cpu()
    del Xg, similarity
    return out

def compare_knn(X, Y, k=10):
    nx, ny = get_neighbors(X, k), get_neighbors(Y, k)
    # Vectorized overlap; exactly the original matching definition.
    return ((nx.unsqueeze(2) == ny.unsqueeze(1)).any(dim=2).float().sum(dim=1) / k).mean().item()

def compare_cka(X, Y):
    # GPU matrix multiplication, with the original centered Gram definition.
    X, Y = X.to(DEVICE), Y.to(DEVICE)
    X = X - X.mean(0, keepdim=True); Y = Y - Y.mean(0, keepdim=True)
    Kx, Ky = X @ X.T, Y @ Y.T
    score = ((Kx * Ky).sum() / (torch.linalg.vector_norm(Kx)*torch.linalg.vector_norm(Ky)).clamp_min(1e-12)).item()
    return score

def compare_rsa(X, Y):
    # Keep scipy Spearman CPU procedure identical to the original experiment.
    X, Y = X.to(DEVICE), Y.to(DEVICE)
    sx = (X @ X.T).cpu().numpy(); sy = (Y @ Y.T).cpu().numpy()
    idx = np.triu_indices(len(X), k=1)
    return float(spearmanr(sx[idx], sy[idx]).statistic)

def matrix_for(reps, names, metric):
    M = np.eye(len(names), dtype=float)
    # symmetric matrix: calculate each pair only once
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            M[i,j] = M[j,i] = metric(reps[names[i]], reps[names[j]])
    return M

def show_matrix(M, names, title, file):
    fig, ax = plt.subplots(figsize=(max(8, len(names)*.85), max(6, len(names)*.72)))
    im=ax.imshow(M, vmin=0, vmax=1, cmap='viridis')
    ax.set_xticks(range(len(names)), names, rotation=55, ha='right')
    ax.set_yticks(range(len(names)), names)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j,i,f'{M[i,j]:.2f}',ha='center',va='center',fontsize=7,color='white' if M[i,j]<.6 else 'black')
    ax.set_title(title); fig.colorbar(im,ax=ax); fig.tight_layout()
    fig.savefig(RESULTS_DIR/file,dpi=140,bbox_inches='tight'); plt.show()

LAYERS = {}
def free_gpu():
    gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()

def add_model(key, repo, loader=AutoModel, prefix='', encoder_only=False, tok_repo=None, model=None, destination=LAYERS):
    tok = AutoTokenizer.from_pretrained(tok_repo or repo)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    if model is None: model = loader.from_pretrained(repo)
    if encoder_only: model = model.get_encoder()
    destination[key] = extract_layers(model, tok, [prefix+t for t in texts])
    print(f'{key:15s} {len(destination[key]):2d} layers; {tuple(destination[key][0].shape)}')
    del model; free_gpu()


## 2 · AG News dataset (balanced 2,000 examples)

In [ ]:
ag = load_dataset('fancyzhx/ag_news', split='train').shuffle(seed=42)
texts, labels, counts = [], [], {c:0 for c in range(4)}
for row in ag:
    c=row['label']
    if counts[c] < N_PER_CLASS:
        texts.append(row['text']); labels.append(c); counts[c]+=1
    if all(x>=N_PER_CLASS for x in counts.values()): break
labels=np.array(labels)
assert len(texts)==4*N_PER_CLASS and len(labels)==len(texts)
subset=np.random.default_rng(42).choice(len(texts), size=min(SUBSET_N,len(texts)),replace=False)
print('Samples:',len(texts),'| Counts:',counts,'| CKA/RSA subset:',len(subset))


## 3 · Primary models and random control

In [ ]:
tok=AutoTokenizer.from_pretrained('EleutherAI/pythia-160m')
tok.pad_token=tok.eos_token
LAYERS['pythia']=extract_layers(AutoModel.from_pretrained('EleutherAI/pythia-160m'),tok,texts)
del tok; free_gpu()
add_model('bert','bert-base-uncased')
torch.manual_seed(0)
add_model('rand_bert_0',None,tok_repo='bert-base-uncased',model=BertModel(BertConfig()))
add_model('minilm','sentence-transformers/all-MiniLM-L6-v2')


## 4 · Baseline model agreement and topic linear probes

The 400-row subset is used for CKA and RSA; the full 2,000 rows are used for kNN overlap. Topic probe training uses a stratified 75%/25% split. Topic labels are used only in the probe, not in representation extraction.

In [ ]:
selected={'Pythia':LAYERS['pythia'][11], 'BERT':LAYERS['bert'][11],
          'MiniLM':LAYERS['minilm'][-1], 'Random BERT':LAYERS['rand_bert_0'][11]}
representations={name:normalize_rep(X) for name,X in selected.items()}
base_names=list(representations)
base_rows=[]
for i,a in enumerate(base_names):
    for b in base_names[i+1:]:
        X,Y=representations[a],representations[b]
        base_rows.append({'Models':f'{a} vs {b}', 'kNN':compare_knn(X,Y),
                          'CKA':compare_cka(X[subset],Y[subset]),'RSA':compare_rsa(X[subset],Y[subset])})
base_results=pd.DataFrame(base_rows); display(base_results.round(4))
tr,te=train_test_split(np.arange(len(labels)),test_size=.25,random_state=42,stratify=labels)
probe_rows=[]
for name,X in selected.items():
    model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1500,C=.1))
    vals=X.float().numpy();model.fit(vals[tr],labels[tr]);probe_rows.append({'model':name,'accuracy':model.score(vals[te],labels[te])})
probe_results=pd.DataFrame(probe_rows);display(probe_results.round(4))
fig,ax=plt.subplots(figsize=(9,4));ax.bar(probe_results.model,probe_results.accuracy)
ax.axhline(.25,ls=':',color='gray',label='chance');ax.set_ylim(0,1);ax.set_ylabel('Held-out topic accuracy');ax.legend()
fig.tight_layout();fig.savefig(RESULTS_DIR/'baseline_probes.png',dpi=150);plt.show()
base_results.to_csv(RESULTS_DIR/'baseline_agreement.csv',index=False)
probe_results.to_csv(RESULTS_DIR/'topic_probes.csv',index=False)


## 5 · Layerwise agreement and probing

Compare equal-index layers (including embedding layer 0), then create the full Pythia–BERT CKA matrix. These layers have matching counts here; this does not imply equivalent processing depth across architectures.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Requires the previous cells: LAYERS, labels, subset, normalize_rep,
# compare_knn, compare_cka, compare_rsa

def rep(name, layer):
    return normalize_rep(LAYERS[name][layer][subset])

n_layers = len(LAYERS["bert"])  # 13 = embeddings + 12 layers

# ---- A. Same-depth agreement: Pythia vs BERT, and Pythia vs Random BERT
rows = []
for l in range(n_layers):
    p, b, r = rep("pythia", l), rep("bert", l), rep("rand_bert_0", l)
    rows.append({
        "layer": l,
        "kNN P-B": compare_knn(p, b), "CKA P-B": compare_cka(p, b), "RSA P-B": compare_rsa(p, b),
        "kNN P-Rand": compare_knn(p, r), "CKA P-Rand": compare_cka(p, r), "RSA P-Rand": compare_rsa(p, r),
    })
sweep = pd.DataFrame(rows).set_index("layer")
print(sweep.round(3).to_string())

# ---- B. CKA between every Pythia layer and every BERT layer
cka_grid = np.zeros((n_layers, n_layers))
for i in range(n_layers):
    p = rep("pythia", i)
    for j in range(n_layers):
        cka_grid[i, j] = compare_cka(p, rep("bert", j))

# ---- C. Linear probe accuracy per layer
tr, te = train_test_split(np.arange(len(labels)), test_size=0.25,
                          random_state=42, stratify=labels)
probe = {}
for name in ["pythia", "bert", "minilm", "rand_bert_0"]:
    accs = []
    for l in range(len(LAYERS[name])):
        X = LAYERS[name][l].float().numpy()
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, C=0.1))
        clf.fit(X[tr], labels[tr])
        accs.append(clf.score(X[te], labels[te]))
    probe[name] = accs

# ---- Plots
fig, ax = plt.subplots(1, 3, figsize=(20, 5))

for m in ["kNN", "CKA", "RSA"]:
    ax[0].plot(sweep.index, sweep[f"{m} P-B"], marker="o", label=f"{m} Pythia-BERT")
    ax[0].plot(sweep.index, sweep[f"{m} P-Rand"], linestyle="--", alpha=0.6, label=f"{m} Pythia-Random")
ax[0].set_title("Agreement by layer (same index)")
ax[0].set_xlabel("Layer"); ax[0].set_ylabel("Score"); ax[0].legend(fontsize=7)

im = ax[1].imshow(cka_grid, origin="lower", vmin=0, vmax=1, cmap="viridis")
ax[1].set_title("CKA: Pythia layer (y) vs BERT layer (x)")
ax[1].set_xlabel("BERT layer"); ax[1].set_ylabel("Pythia layer")
plt.colorbar(im, ax=ax[1])

for name, accs in probe.items():
    ax[2].plot(range(len(accs)), accs, marker="o", label=name)
ax[2].axhline(0.25, color="gray", linestyle=":", label="chance")
ax[2].set_title("Topic probe accuracy by layer")
ax[2].set_xlabel("Layer"); ax[2].set_ylabel("Accuracy"); ax[2].legend()

plt.tight_layout()
plt.show()
sweep.to_csv(RESULTS_DIR/'layerwise_agreement.csv')
pd.DataFrame(cka_grid).to_csv(RESULTS_DIR/'cka_grid.csv',index=False)
fig.savefig(RESULTS_DIR/'layerwise.png',dpi=150,bbox_inches='tight')


## 6 · Additional pretrained encoder/decoder models

Models and prefixes match the source study, including the `query: ` prefix for E5. Longformer is relatively expensive and the source experiment includes it. Models are loaded sequentially to minimize peak VRAM.

In [ ]:
EXTRA = [
    ('t5_enc','google-t5/t5-small',T5EncoderModel,'',False),
    ('bart_enc','facebook/bart-base',AutoModel,'',True),
    ('electra','google/electra-small-discriminator',AutoModel,'',False),
    ('distilbert','distilbert/distilbert-base-uncased',AutoModel,'',False),
    ('e5','intfloat/e5-small-v2',AutoModel,'query: ',False),
    ('longformer','allenai/longformer-base-4096',AutoModel,'',False),
]
for key,repo,loader,prefix,encoder_only in EXTRA:
    add_model(key,repo,loader=loader,prefix=prefix,encoder_only=encoder_only)


## 7 · Expanded cross-model matrices

At 90% relative depth, compare all available standard pretrained models. We do not average kNN, CKA and RSA into a single scientific similarity score; each metric is presented separately.

In [ ]:
preferred=['pythia','bert','distilbert','electra','t5_enc','bart_enc','minilm','e5','longformer','rand_bert_0']
names=[m for m in preferred if m in LAYERS]
reps={m:normalize_rep(LAYERS[m][round(DEPTH*(len(LAYERS[m])-1))][subset]) for m in names}
mats={'kNN':matrix_for(reps,names,compare_knn),
      'CKA':matrix_for(reps,names,compare_cka),
      'RSA':matrix_for(reps,names,compare_rsa)}
for metric,M in mats.items():
    pd.DataFrame(M,index=names,columns=names).to_csv(RESULTS_DIR/f'matrix_{metric}.csv')
    show_matrix(M,names,f'{metric} at relative depth {DEPTH}',f'matrix_{metric}.png')


## 8 · Optional: autoregressive vs. diffusion nano models

The original notebook clones `tutao0123/nanoDiffusionLab` and downloads `TuTao/nanoDiffusionLab-TinyStories-106M` weights. This is **external executable code**, rather than a normal Hugging Face Transformer model. It is intentionally **opt-in**; inspect its source and checkpoint before running. Tokens are processed one sentence at a time to prevent bidirectional padding leakage. This preserves the original experiment's semantics, though it does not maximize GPU throughput.

In [ ]:
if not INCLUDE_NANO:
    print('Nano AR vs diffusion: skipped (set INCLUDE_NANO=True in Setup to enable).')
else:
    import json
    import os
    import subprocess
    import sys

    import numpy as np
    import pandas as pd
    import torch
    import matplotlib.pyplot as plt
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    from transformers import AutoTokenizer
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression

    # Requires the setup cell (LAYERS, texts, labels, device) and the GPT cell
    # (subset, normalize_rep, compare_knn, compare_cka, compare_rsa)

    if not os.path.isdir("nanoDiffusionLab"):
        subprocess.run(["git", "clone", "-q", "https://github.com/tutao0123/nanoDiffusionLab"], check=True)
    sys.path.insert(0, os.path.abspath("nanoDiffusionLab"))
    from config import ModelConfig
    from model import Transformer

    REPO = "TuTao/nanoDiffusionLab-TinyStories-106M"
    gpt2 = AutoTokenizer.from_pretrained("gpt2")


    def load_nano(prefix):
        cfg = json.load(open(hf_hub_download(REPO, f"{prefix}_config.json")))["model"]
        model = Transformer(ModelConfig(**cfg))
        state = load_file(hf_hub_download(REPO, f"{prefix}.safetensors"))
        model.load_state_dict({k: v.float() for k, v in state.items()})
        return model.to(device).eval()


    @torch.no_grad()
    def nano_layers(model, texts, max_len=64):
        # One sentence at a time: the diffusion model is bidirectional, so padding
        # tokens would leak into every position.
        per_layer = None
        for text in texts:
            ids = gpt2(text, truncation=True, max_length=max_len, return_tensors="pt")["input_ids"].to(device)
            positions = torch.arange(ids.shape[1], device=device)
            x = model.token_embedding(ids) + model.position_embedding(positions)
            if model.time_mlp is not None:
                t = ids.eq(model.config.mask_token_id).float().mean(dim=1)  # 0: nothing masked
                x = x + model.time_mlp(t[:, None])[:, None, :]
            states = [x]
            for block in model.blocks:
                x = block(x)
                states.append(x)
            pooled = [s.mean(dim=1).cpu() for s in states]
            if per_layer is None:
                per_layer = [[] for _ in pooled]
            for l, p in enumerate(pooled):
                per_layer[l].append(p)
        return [torch.cat(x) for x in per_layer]


    for key, prefix in [("nano_ar", "ar"), ("nano_mdlm", "mdlm")]:
        model = load_nano(prefix)
        LAYERS[key] = nano_layers(model, texts)
        print(key, "layers:", len(LAYERS[key]), "dim:", LAYERS[key][0].shape[1])
        del model
        torch.cuda.empty_cache()


    def rep(name, layer):
        return normalize_rep(LAYERS[name][layer][subset])


    n_layers = len(LAYERS["nano_ar"])

    rows = []
    for l in range(n_layers):
        a, d = rep("nano_ar", l), rep("nano_mdlm", l)
        rows.append({"layer": l,
                     "kNN": compare_knn(a, d), "CKA": compare_cka(a, d), "RSA": compare_rsa(a, d)})
    agree = pd.DataFrame(rows).set_index("layer")
    print("AR vs Diffusion (same architecture, same data, same layer):")
    print(agree.round(3).to_string())

    tr, te = train_test_split(np.arange(len(labels)), test_size=0.25, random_state=42, stratify=labels)
    probe = {}
    for name in ["nano_ar", "nano_mdlm"]:
        accs = []
        for l in range(n_layers):
            X = LAYERS[name][l].float().numpy()
            clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, C=0.1))
            clf.fit(X[tr], labels[tr])
            accs.append(clf.score(X[te], labels[te]))
        probe[name] = accs

    other = [k for k in ["pythia", "bert", "rand_bert_0"] if k in LAYERS]
    cross = {}
    for name in ["nano_ar", "nano_mdlm"]:
        a = rep(name, n_layers - 2)
        cross[name] = {o: (compare_knn(a, rep(o, len(LAYERS[o]) - 2)),
                           compare_cka(a, rep(o, len(LAYERS[o]) - 2)),
                           compare_rsa(a, rep(o, len(LAYERS[o]) - 2))) for o in other}
    print("\nAgreement with other models (second-to-last layer; kNN, CKA, RSA):")
    for name, d in cross.items():
        for o, v in d.items():
            print(f"{name:10s} vs {o:12s} " + "  ".join(f"{x:.3f}" for x in v))

    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    for m in ["kNN", "CKA", "RSA"]:
        ax[0].plot(agree.index, agree[m], marker="o", label=m)
    ax[0].set_title("nano AR vs nano Diffusion, by layer")
    ax[0].set_xlabel("Layer"); ax[0].set_ylabel("Score"); ax[0].legend()
    for name, accs in probe.items():
        ax[1].plot(range(len(accs)), accs, marker="o", label=name)
    ax[1].axhline(0.25, color="gray", linestyle=":", label="chance")
    ax[1].set_title("Topic probe accuracy by layer")
    ax[1].set_xlabel("Layer"); ax[1].set_ylabel("Accuracy"); ax[1].legend()
    plt.tight_layout()
    plt.show()


## 9 · Neighbour consensus across multiple models

Test whether a neighbour selected by two models is more likely to be selected by a third, whether some sentences induce systematically higher agreement, and how agreement relates to news topic and lexical Jaccard overlap. Neighbour masks are **cached once** and reused instead of recomputing the same 2,000×2,000 similarity matrices.

In [ ]:
import itertools

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# Requires: setup cell (LAYERS, texts), GPT cell (normalize_rep, get_neighbors),
# and optionally more_models (extra models). Runs on the news sentences.

K = 10
DEPTH = 0.9
candidates = ["pythia", "bert", "distilbert", "electra", "t5_enc", "bart_enc",
              "minilm", "e5", "longformer"]
real = [n for n in candidates if n in LAYERS]
print("Models:", real)


def neighbor_mask(key):
    X=normalize_rep(LAYERS[key][round(DEPTH*(len(LAYERS[key])-1))])
    idx=get_neighbors(X,K)
    mask=torch.zeros(len(X),len(X),dtype=torch.bool)
    mask.scatter_(1,idx,True)
    return mask

masks = {n: neighbor_mask(n) for n in real}
if "rand_bert_0" in LAYERS:
    masks["rand_bert_0"] = neighbor_mask("rand_bert_0")
n = len(texts)
chance = K / (n - 1)

# 1. If A and B agree that j is a neighbor of i, does a third model C also list j?
ratios, p_cond_all, p_base_all = [], [], []
for A, B, C in itertools.permutations(real, 3):
    if A > B:
        continue
    both = masks[A] & masks[B]
    p_cond = ((both & masks[C]).sum() / both.sum()).item()
    p_base = ((masks[A] & masks[C]).sum() / masks[A].sum()).item()
    p_cond_all.append(p_cond)
    p_base_all.append(p_base)

print("\n1. Does agreement between two models predict a third model's agreement?")
print(f"   P(C lists j | A lists j)            = {np.mean(p_base_all):.3f}")
print(f"   P(C lists j | A and B both list j)  = {np.mean(p_cond_all):.3f}")
print(f"   Chance (random neighbor)            = {chance:.3f}")
if "rand_bert_0" in masks:
    both_rand = [((masks[A] & masks[B]) & masks["rand_bert_0"]).sum().item() /
                 (masks[A] & masks[B]).sum().item() for A, B in itertools.combinations(real, 2)]
    print(f"   P(Random BERT lists j | A and B both list j) = {np.mean(both_rand):.3f}")

# 2. For a neighbor listed by one model, how many OTHER models list it too?
others_hist = np.zeros(len(real))
for A in real:
    support = sum(masks[B].int() for B in real if B != A)
    counts = support[masks[A]]
    others_hist += np.bincount(counts.numpy(), minlength=len(real))[:len(real)] / counts.numel()
others_hist /= len(real)

# 3. Are the same sentences "easy" for different pairs of models?
pair_scores = {(A, B): ((masks[A] & masks[B]).sum(1).float() / K).numpy()
               for A, B in itertools.combinations(real, 2)}
share, disjoint = [], []
for p, q in itertools.combinations(pair_scores, 2):
    r = spearmanr(pair_scores[p], pair_scores[q]).statistic
    (share if set(p) & set(q) else disjoint).append(r)
print("\n3. Correlation of per-sentence agreement between two model pairs")
print(f"   pairs sharing a model: {np.mean(share):.3f}")
print(f"   disjoint pairs:        {np.mean(disjoint):.3f}")

mean_agree = np.mean(list(pair_scores.values()), axis=0)
order = np.argsort(mean_agree)
print("\nSentences where models agree MOST on neighbors:")
for i in order[::-1][:5]:
    print(f"  {mean_agree[i]:.2f}  {texts[i][:110]}")
print("\nSentences where models agree LEAST:")
for i in order[:5]:
    print(f"  {mean_agree[i]:.2f}  {texts[i][:110]}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].bar(range(len(real)), others_hist)
ax[0].set_xlabel(f"Number of OTHER models (out of {len(real) - 1}) that list the same neighbor")
ax[0].set_ylabel("Fraction of neighbors")
ax[0].set_title("How widely shared is a model's neighbor?")
ax[1].hist(mean_agree, bins=30)
ax[1].set_xlabel("Mean pairwise kNN overlap for the sentence")
ax[1].set_ylabel("Number of sentences")
ax[1].set_title("Are some sentences easier to agree on?")
plt.tight_layout()
plt.show()

fig.savefig(RESULTS_DIR/'neighbor_consensus.png',dpi=150,bbox_inches='tight')


In [ ]:
import re

import numpy as np
import torch
import matplotlib.pyplot as plt

# Requires: setup cell (LAYERS, texts, labels), GPT cell (normalize_rep, get_neighbors)

K = 10
DEPTH = 0.9
candidates = ["pythia", "bert", "distilbert", "electra", "t5_enc", "bart_enc",
              "minilm", "e5", "longformer"]
real = [n for n in candidates if n in LAYERS]
labels_np = np.asarray(labels)
n = len(texts)
words = [set(re.findall(r"[a-z0-9']+", t.lower())) for t in texts]


def jaccard(i, j):
    a, b = words[i], words[j]
    return len(a & b) / max(1, len(a | b))


# Reuse cached masks constructed in the previous section.
assert all(m in masks for m in real)
max_support = len(real) - 1

same = np.zeros(max_support + 1)
jac = np.zeros(max_support + 1)
count = np.zeros(max_support + 1)

for A in real:
    support = sum(masks[B].int() for B in real if B != A)
    rows, cols = masks[A].nonzero(as_tuple=True)
    rows, cols = rows.numpy(), cols.numpy()
    s = support[rows, cols].numpy()
    same_label = labels_np[rows] == labels_np[cols]
    jac_vals = np.fromiter((jaccard(i, j) for i, j in zip(rows, cols)), dtype=float, count=len(rows))
    for level in range(max_support + 1):
        sel = s == level
        same[level] += same_label[sel].sum()
        jac[level] += jac_vals[sel].sum()
        count[level] += sel.sum()

frac_same = same / np.maximum(count, 1)
mean_jac = jac / np.maximum(count, 1)

print("Number of OTHER models that also list the neighbor | same-topic rate | word overlap | neighbors")
for level in range(max_support + 1):
    print(f"{level:>2}  {frac_same[level]:.3f}  {mean_jac[level]:.3f}  {int(count[level])}")

baseline = np.mean([(labels_np == c).mean() ** 2 for c in np.unique(labels_np)])
print(f"\nSame-topic rate for a random pair: {baseline:.3f}")

if "rand_bert_0" in LAYERS:
    r = neighbor_mask("rand_bert_0")
    rows, cols = r.nonzero(as_tuple=True)
    print("Random BERT neighbors same-topic rate:",
          round(float((labels_np[rows.numpy()] == labels_np[cols.numpy()]).mean()), 3))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(range(max_support + 1), frac_same, marker="o")
ax[0].axhline(baseline, color="gray", linestyle=":", label="random pair")
ax[0].set_xlabel("Number of other models that agree on the neighbor")
ax[0].set_ylabel("Fraction with the same topic")
ax[0].set_title("Do agreed neighbors share a topic?")
ax[0].set_ylim(0, 1)
ax[0].legend()
ax[1].plot(range(max_support + 1), mean_jac, marker="o", color="tab:orange")
ax[1].set_xlabel("Number of other models that agree on the neighbor")
ax[1].set_ylabel("Mean word overlap (Jaccard)")
ax[1].set_title("Do agreed neighbors share words?")
plt.tight_layout()
plt.show()

fig.savefig(RESULTS_DIR/'neighbor_semantics.png',dpi=150,bbox_inches='tight')


## 10 · Stability over subsamples (ten seeds)

Like the original notebook, use ten randomly selected 1,000-text subsets (400 for CKA/RSA). These repeated subsets are *not* independent test datasets; the reported standard deviation describes sensitivity to sampling from the same 2,000 documents. Aggregate score is only an exploratory ranking, not a canonical metric.

In [ ]:
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Requires: setup cell (LAYERS, texts), GPT cell (normalize_rep, compare_knn,
# compare_cka, compare_rsa). Models missing from LAYERS are skipped.

SEEDS = list(range(10))
N_SUBSET = 1000
N_CKA = 400
DEPTH = 0.9

preferred = ["pythia", "nano_ar", "nano_mdlm", "bert", "distilbert", "electra",
             "t5_enc", "bart_enc", "minilm", "e5", "longformer", "rand_bert_0"]
names = [m for m in preferred if m in LAYERS]
n = len(texts)

reps = {}
for m in names:
    layer = round(DEPTH * (len(LAYERS[m]) - 1))
    reps[m] = normalize_rep(LAYERS[m][layer])

pairs = list(itertools.combinations(names, 2))
scores = {(p, metric): [] for p in pairs for metric in ("kNN", "CKA", "RSA")}
avg_by_seed = {m: [] for m in names}

for seed in SEEDS:
    rng = np.random.default_rng(seed)
    sub = rng.choice(n, size=N_SUBSET, replace=False)
    sub_small = sub[:N_CKA]
    per_model = {m: [] for m in names}
    for a, b in pairs:
        knn = compare_knn(reps[a][sub], reps[b][sub])
        cka = compare_cka(reps[a][sub_small], reps[b][sub_small])
        rsa = compare_rsa(reps[a][sub_small], reps[b][sub_small])
        scores[((a, b), "kNN")].append(knn)
        scores[((a, b), "CKA")].append(cka)
        scores[((a, b), "RSA")].append(rsa)
        mean_score = (knn + cka + rsa) / 3
        per_model[a].append(mean_score)
        per_model[b].append(mean_score)
    for m in names:
        avg_by_seed[m].append(np.mean(per_model[m]))
    print(f"seed {seed} done")

rows = []
for (a, b) in pairs:
    row = {"pair": f"{a} vs {b}"}
    for metric in ("kNN", "CKA", "RSA"):
        v = np.array(scores[((a, b), metric)])
        row[metric] = f"{v.mean():.3f} +- {v.std():.3f}"
        row[f"{metric}_std"] = v.std()
    rows.append(row)
table = pd.DataFrame(rows)
print("\nPairs with the largest variation across seeds:")
print(table.sort_values("kNN_std", ascending=False)[["pair", "kNN", "CKA", "RSA"]].head(8).to_string(index=False))
print("\nTypical std across seeds: kNN %.4f, CKA %.4f, RSA %.4f" % (
    table["kNN_std"].mean(), table["CKA_std"].mean(), table["RSA_std"].mean()))

matrix = np.array([avg_by_seed[m] for m in names])
ranks = (-matrix).argsort(axis=0).argsort(axis=0) + 1
print("\nModel ranking by average agreement (1 = closest to the others):")
summary = pd.DataFrame({
    "model": names,
    "mean score": matrix.mean(axis=1).round(3),
    "std": matrix.std(axis=1).round(3),
    "mean rank": ranks.mean(axis=1).round(2),
    "rank std": ranks.std(axis=1).round(2),
}).sort_values("mean rank")
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
order = summary["model"].tolist()
means = [np.mean(avg_by_seed[m]) for m in order]
stds = [np.std(avg_by_seed[m]) for m in order]
ax.bar(order, means, yerr=stds, capsize=4)
ax.set_ylabel("Average agreement (kNN, CKA, RSA)")
ax.set_title(f"Mean +- std over {len(SEEDS)} random sentence samples")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

summary.to_csv(RESULTS_DIR/'resampling_summary.csv',index=False)
fig.savefig(RESULTS_DIR/'resampling.png',dpi=150,bbox_inches='tight')


## 11 · Optional: TinyStories dataset replication

This uses 1,000 TinyStories validation snippets and loads models again, to test whether AG News results persist on another domain. This section is optional because it downloads weights and extracts new representations. If `INCLUDE_NANO=True`, both nano models also run.

In [ ]:
if not RUN_TINYSTORIES:
    print('TinyStories replication: skipped (set RUN_TINYSTORIES=True in Setup to enable).')
else:
    import gc

    import numpy as np
    import torch
    import matplotlib.pyplot as plt
    from datasets import load_dataset
    from transformers import (AutoModel, AutoTokenizer, BartModel, BertConfig,
                              BertModel, T5EncoderModel)

    # Requires: setup cell (extract_layers, device), GPT cell (normalize_rep,
    # compare_knn, compare_cka, compare_rsa) and diffusion_models cell
    # (load_nano, nano_layers)

    ts_ds = load_dataset("roneneldan/TinyStories", split="validation").shuffle(seed=0)
    ts_texts = []
    for row in ts_ds:
        t = row["text"].strip().replace("\n", " ")[:250]
        if len(t) > 40:
            ts_texts.append(t)
        if len(ts_texts) == 1000:
            break
    print(len(ts_texts), "TinyStories snippets. Example:", ts_texts[0][:120])

    LAYERS_TS = {}


    def run_hf(key, repo, loader=AutoModel, prefix="", encoder_only=False, tok_repo=None, model=None):
        tok = AutoTokenizer.from_pretrained(tok_repo or repo)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        if model is None:
            model = loader.from_pretrained(repo)
        if encoder_only:
            model = model.get_encoder()
        LAYERS_TS[key] = extract_layers(model, tok, [prefix + t for t in ts_texts])
        print(key, len(LAYERS_TS[key]), "layers")
        del model
        gc.collect()
        torch.cuda.empty_cache()


    torch.manual_seed(0)
    run_hf("pythia", "EleutherAI/pythia-160m")
    run_hf("bert", "bert-base-uncased")
    run_hf("rand_bert_0", None, tok_repo="bert-base-uncased", model=BertModel(BertConfig()))
    run_hf("distilbert", "distilbert/distilbert-base-uncased")
    run_hf("electra", "google/electra-small-discriminator")
    run_hf("t5_enc", "google-t5/t5-small", loader=T5EncoderModel)
    run_hf("bart_enc", "facebook/bart-base", loader=BartModel, encoder_only=True)
    run_hf("minilm", "sentence-transformers/all-MiniLM-L6-v2")
    run_hf("e5", "intfloat/e5-small-v2", prefix="query: ")

    for key, prefix in ([("nano_ar", "ar"), ("nano_mdlm", "mdlm")] if INCLUDE_NANO else []):
        model = load_nano(prefix)
        LAYERS_TS[key] = nano_layers(model, ts_texts)
        print(key, len(LAYERS_TS[key]), "layers")
        del model
        torch.cuda.empty_cache()

    ts_subset = np.random.default_rng(42).choice(len(ts_texts), size=400, replace=False)
    DEPTH = 0.9
    preferred = ["pythia", "nano_ar", "nano_mdlm", "bert", "distilbert", "electra",
                 "t5_enc", "bart_enc", "minilm", "e5", "rand_bert_0"]
    names = [n for n in preferred if n in LAYERS_TS]


    def ts_rep(key, layer=None):
        n = len(LAYERS_TS[key])
        layer = round(DEPTH * (n - 1)) if layer is None else layer
        return normalize_rep(LAYERS_TS[key][layer][ts_subset])


    reps = {k: ts_rep(k) for k in names}


    def matrix(fn):
        M = np.eye(len(names))
        for i, a in enumerate(names):
            for j, b in enumerate(names):
                if i != j:
                    M[i, j] = fn(reps[a], reps[b])
        return M


    mats = {"kNN": matrix(compare_knn), "CKA": matrix(compare_cka), "RSA": matrix(compare_rsa)}

    fig, axes = plt.subplots(1, 3, figsize=(24, 7))
    for ax, (title, M) in zip(axes, mats.items()):
        im = ax.imshow(M, vmin=0, vmax=1, cmap="viridis")
        ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=60, ha="right")
        ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
        for i in range(len(names)):
            for j in range(len(names)):
                ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", color="white", fontsize=7)
        ax.set_title(f"TinyStories - {title} (depth {DEPTH})")
        plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()

    print("Average agreement with all other models (TinyStories text):")
    scores = {}
    for i, n in enumerate(names):
        others = [j for j in range(len(names)) if j != i]
        scores[n] = np.mean([mats[m][i, others].mean() for m in mats])
    for n, v in sorted(scores.items(), key=lambda x: -x[1]):
        print(f"{n:12s} {v:.3f}")

    print("\nnano AR vs nano Diffusion by layer (TinyStories text):")
    for l in range(len(LAYERS_TS.get("nano_ar", []))):
        a, d = ts_rep("nano_ar", l), ts_rep("nano_mdlm", l)
        print(f"layer {l:2d}  kNN {compare_knn(a, d):.3f}  CKA {compare_cka(a, d):.3f}  RSA {compare_rsa(a, d):.3f}")

    print("\nnano models vs BERT / Pythia / Random BERT (kNN, CKA, RSA):")
    for nm in [nm for nm in ["nano_ar", "nano_mdlm"] if nm in reps]:
        for o in ["pythia", "bert", "rand_bert_0"]:
            a, b = reps[nm], reps[o]
            print(f"{nm:10s} vs {o:12s} {compare_knn(a, b):.3f}  {compare_cka(a, b):.3f}  {compare_rsa(a, b):.3f}")


## 12 · External semantic benchmarks: STS-B and PAWS

STS-B: Spearman rank correlation between cosine similarity and human similarity ratings. PAWS: ROC-AUC for detecting paraphrases. As in the original notebook, all layers are evaluated and both a fixed 90%-depth layer and the *best observed layer* are reported. **The best layer is selected on the same test set, so treat that number as exploratory/optimistically biased, not a held-out evaluation.**

In [ ]:
import gc
import re

import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from transformers import (AutoModel, AutoTokenizer, BartModel, BertConfig,
                          BertModel, T5EncoderModel)

# Requires: setup cell (extract_layers, device), GPT cell (normalize_rep).
# Optional: diffusion_models cell (load_nano, nano_layers) to include the nano models.

sts = load_dataset("sentence-transformers/stsb", split="test")
sts_a, sts_b = list(sts["sentence1"]), list(sts["sentence2"])
sts_gold = np.array(sts["score"])

paws = load_dataset("google-research-datasets/paws", "labeled_final", split="test").shuffle(seed=0).select(range(2000))
paws_a, paws_b = list(paws["sentence1"]), list(paws["sentence2"])
paws_label = np.array(paws["label"])

n_sts, n_paws = len(sts_a), len(paws_a)
all_texts = sts_a + sts_b + paws_a + paws_b
slices = {
    "sts": slice(0, 2 * n_sts),
    "paws": slice(2 * n_sts, 2 * n_sts + 2 * n_paws),
}
print(n_sts, "STS-B pairs,", n_paws, "PAWS pairs")


def tokens(t):
    return set(re.findall(r"[a-z0-9']+", t.lower()))


def jaccard(a, b):
    x, y = tokens(a), tokens(b)
    return len(x & y) / max(1, len(x | y))


baseline_sts = spearmanr([jaccard(a, b) for a, b in zip(sts_a, sts_b)], sts_gold).statistic
baseline_paws = roc_auc_score(paws_label, [jaccard(a, b) for a, b in zip(paws_a, paws_b)])
print(f"Word-overlap baseline: STS-B Spearman {baseline_sts:.3f}, PAWS AUC {baseline_paws:.3f}")


def evaluate(layers):
    sts_scores, paws_scores = [], []
    for L in layers:
        X = normalize_rep(L[slices["sts"]])
        cos = (X[:n_sts] * X[n_sts:]).sum(1).numpy()
        sts_scores.append(spearmanr(cos, sts_gold).statistic)
        X = normalize_rep(L[slices["paws"]])
        cos = (X[:n_paws] * X[n_paws:]).sum(1).numpy()
        paws_scores.append(roc_auc_score(paws_label, cos))
    return sts_scores, paws_scores


results = {}


def run_hf(key, repo, loader=AutoModel, prefix="", encoder_only=False, tok_repo=None, model=None):
    tok = AutoTokenizer.from_pretrained(tok_repo or repo)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if model is None:
        model = loader.from_pretrained(repo)
    if encoder_only:
        model = model.get_encoder()
    layers = extract_layers(model, tok, [prefix + t for t in all_texts])
    results[key] = evaluate(layers)
    print(key, "done")
    del model, layers
    gc.collect()
    torch.cuda.empty_cache()


torch.manual_seed(0)
run_hf("pythia", "EleutherAI/pythia-160m")
run_hf("bert", "bert-base-uncased")
run_hf("rand_bert_0", None, tok_repo="bert-base-uncased", model=BertModel(BertConfig()))
run_hf("distilbert", "distilbert/distilbert-base-uncased")
run_hf("electra", "google/electra-small-discriminator")
run_hf("t5_enc", "google-t5/t5-small", loader=T5EncoderModel)
run_hf("bart_enc", "facebook/bart-base", loader=BartModel, encoder_only=True)
run_hf("minilm", "sentence-transformers/all-MiniLM-L6-v2")
run_hf("e5", "intfloat/e5-small-v2", prefix="query: ")

if INCLUDE_NANO and "load_nano" in globals() and "nano_layers" in globals():
    for key, prefix in [("nano_ar", "ar"), ("nano_mdlm", "mdlm")]:
        model = load_nano(prefix)
        layers = nano_layers(model, all_texts)
        results[key] = evaluate(layers)
        print(key, "done")
        del model, layers
        torch.cuda.empty_cache()

print("\nModel | STS-B Spearman (layer at 90% depth / best layer) | PAWS AUC (90% depth / best layer)")
for key, (s, p) in results.items():
    d = round(0.9 * (len(s) - 1))
    print(f"{key:12s} {s[d]:.3f} / {max(s):.3f} (layer {int(np.argmax(s))})   "
          f"{p[d]:.3f} / {max(p):.3f} (layer {int(np.argmax(p))})")
print(f"{'word overlap':12s} {baseline_sts:.3f}                       {baseline_paws:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for key, (s, p) in results.items():
    x = np.linspace(0, 1, len(s))
    ax[0].plot(x, s, marker="o", label=key)
    ax[1].plot(x, p, marker="o", label=key)
ax[0].axhline(baseline_sts, color="gray", linestyle=":", label="word overlap")
ax[0].set_title("STS-B: similarity vs human judgment (Spearman)")
ax[1].axhline(baseline_paws, color="gray", linestyle=":", label="word overlap")
ax[1].axhline(0.5, color="black", linestyle="--", linewidth=0.8)
ax[1].set_title("PAWS: paraphrase vs word-swap (AUC, 0.5 = chance)")
for a in ax:
    a.set_xlabel("Relative depth (0 = embeddings, 1 = last layer)")
ax[0].set_ylabel("Spearman")
ax[1].set_ylabel("AUC")
ax[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

fig.savefig(RESULTS_DIR/'sts_paws.png',dpi=150,bbox_inches='tight')
pd.DataFrame([{'model':k,'sts_90pct':v[0][round(.9*(len(v[0])-1))],'sts_best_observed':max(v[0]),'paws_90pct':v[1][round(.9*(len(v[1])-1))],'paws_best_observed':max(v[1])} for k,v in results.items()]).to_csv(RESULTS_DIR/'sts_paws.csv',index=False)


## 13 · Export results

Download a ZIP containing numeric outputs and generated figures. Colab stores temporary files until you download them; no external credentials are required.

In [ ]:
import shutil
zip_path=shutil.make_archive('cross_model_analysis_results','zip',root_dir=RESULTS_DIR)
print('Result archive:',zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not running in Colab: download the ZIP from the working directory.')


## Interpretation checklist and limitations

- Larger kNN overlap indicates a more similar neighbourhood ordering, **not identical semantics**.
- The AG News dataset includes lexical/topic structure; a lexical baseline and topic-conditioned analysis help but do not fully isolate semantic information.
- Randomly initialized BERT is a limited control and does not represent all untrained architectures.
- Most models have different depths and objectives; **90% relative depth** is a heuristic.
- Correlations computed on pairwise similarities reuse the same texts, so pairs are not independent observations.
- Linear-probe labels are evaluated on a holdout, but model/layer selection done after looking at results can introduce selection bias.
- External checkpoints and Hugging Face package versions can change; keep the executed notebook and outputs for provenance.
- **STS-B/PAWS best layer** is descriptive because it is chosen using the test labels themselves. For confirmatory claims, select a layer on a validation split and assess on a disjoint test split.
- Resampling seeds vary *samples*, not pretrained model initialization or independent training runs.

### Research interpretation

After the run, use the printed tables and figures to write results grounded in the observed numbers. Do not copy old exploratory conclusions without rechecking them against this newly executed notebook.
